# 🧼 Clean VCT Agent Composition Data by Year (2021–2025)
This notebook loads yearly `teams_picked_agents.csv` files, cleans them, and combines into one master file.

## 📦 Step 1: Import Required Libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
import numpy as np

## 📁 Step 2: Define Paths and Prepare Output

In [ ]:
# Set the root path where yearly folders are located
root_path = os.path.join("kaggle", "input", "valorant-champion-tour-2021-2023-data")
output_dir = 'cleaned_data'
os.makedirs(output_dir, exist_ok=True)
merged_all_years = []

## 🧹 Step 3: Clean and Save Data by Year

In [ ]:
# Loop through folders and clean
for folder_name in os.listdir(root_path):
    if folder_name.startswith('vct_') and folder_name[4:].isdigit():
        year = folder_name[-4:]
        folder_path = os.path.join(root_path, folder_name)
        file_path = os.path.join(folder_path, 'agents', 'teams_picked_agents.csv')

        if not os.path.exists(file_path):
            print(f"Skipping missing file: {file_path}")
            continue

        data = pd.read_csv(file_path)
        data['Win Rate'] = data['Total Wins By Map'] / data['Total Maps Played']
        data['Win Rate'] = data['Win Rate'].fillna(0)
        data['Win'] = (data['Total Wins By Map'] > data['Total Loss By Map']).astype(int)

        cleaned_data = (
            data.groupby(['Tournament', 'Stage', 'Match Type', 'Map', 'Team'])
            .agg({
                'Agent': lambda x: ', '.join(sorted(x)),
                'Win': 'max'
            })
            .reset_index()
        )

        cleaned_data.rename(columns={'Agent': 'Agents Played', 'Win': 'Match Outcome'}, inplace=True)
        cleaned_data['Year'] = int(year)
        merged_all_years.append(cleaned_data)

        output_file = os.path.join(output_dir, f'cleaned_match_data_{year}.csv')
        cleaned_data.to_csv(output_file, index=False)
        print(f"✅ Saved: {output_file}")

## 📦 Step 4: Combine All Years into One Master File

In [ ]:
if merged_all_years:
    all_years_df = pd.concat(merged_all_years, ignore_index=True)
    master_path = os.path.join(output_dir, 'cleaned_match_data_all_years.csv')
    all_years_df.to_csv(master_path, index=False)
    print(f"🎉 Combined file saved to: {master_path}")
else:
    print("⚠️ No data was processed.")